In [8]:
!mv /content/*count_file* /content/GSE150910_gene_level_count_file.csv


In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import seaborn as sns

df = pd.read_csv("/content/GSE150910_gene_level_count_file.csv", index_col=0, compression="gzip")
samples = df.columns.values
genes = df.index.values


In [14]:
labels = np.array([s.split("_")[0] for s in samples])
keep_samples = (labels == "control") | (labels == "ipf")
df = df.iloc[:, keep_samples]
labels = labels[keep_samples]


In [15]:
df_cpm = df.div(df.sum(axis=0), axis=1) * 1_000_000


In [16]:
mean_control = df_cpm.iloc[:, labels == "control"].mean(axis=1)
mean_ipf = df_cpm.iloc[:, labels == "ipf"].mean(axis=1)
df_filtered = df_cpm[(mean_control >= 5) | (mean_ipf >= 5)]


In [20]:
p_values = []
log2_fc = []
epsilon = 1

for gene in df_filtered.index:
    control_vals = df_filtered.loc[gene, labels == "control"]
    ipf_vals = df_filtered.loc[gene, labels == "ipf"]
    _, p_val = ks_2samp(control_vals, ipf_vals)
    fc = (ipf_vals.mean() + epsilon) / (control_vals.mean() + epsilon)

    p_values.append(p_val)
    log2_fc.append(np.log2(fc))


In [21]:
results_df = pd.DataFrame({"p_value": p_values, "log2_FC": log2_fc}, index=df_filtered.index)
results_df["p_value_bonf"] = results_df["p_value"] * len(results_df)

significant_up = results_df[(results_df["p_value_bonf"] <= 0.05) & (results_df["log2_FC"] >= 2)]
significant_up.to_csv("upregulated_DEGs.csv")
